In [1]:
import pandas as pd 
import json
import mne
import numpy as np
import mne
import os
from itertools import product
import glob

# --------------------------------------------------------------------------
# REPRODUCIBILITY & HARDWARE SETUP (Must be first)
# ---------------------------------------------------------------------------
print("it started")
import os
import random
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

import numpy as np
import tensorflow as tf
import torch

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.config.threading.set_inter_op_parallelism_threads(1)
tf.config.threading.set_intra_op_parallelism_threads(1)

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

# ---------------------------------------------------------------------------
# ORIGINAL IMPORTS & SETUP
# ---------------------------------------------------------------------------
import json
import uuid
import pandas as pd
import matplotlib.pyplot as plt

# Scipy & MNE
import mne
from scipy.signal import stft, welch
from scipy.stats import entropy, norm
from sklearn.model_selection import KFold, train_test_split

# Scikit-learn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.utils.class_weight import compute_class_weight

# TensorFlow / Keras
from tensorflow.keras import layers, models, Model, callbacks

print(f"Reproducibility settings locked with SEED: {SEED}")

# GPU Check
if tf.config.list_physical_devices('GPU'):
    print("TensorFlow GPU Accelerated Backend Active.")
else:
    print("No GPU detected for TensorFlow. Using CPU.")

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"CUDA GPU Accelerated Backend Active: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")

it started
Reproducibility settings locked with SEED: 42
TensorFlow GPU Accelerated Backend Active.
CUDA GPU Accelerated Backend Active: Tesla T4


In [2]:
tsv_path = "/kaggle/input/datasets/adithyarajnarayanan/pd-eeg-dataset-2/PD EEG dataset/participants.tsv"

df = pd.read_csv(tsv_path, sep="\t")
print(df.head())

  participant_id GROUP    ID     EEG  AGE GENDER  MOCA  UPDRS  TYPE
0        sub-001    PD  1001  PD1001   80      M    19   28.0     1
1        sub-002    PD  1011  PD1011   81      M    17   25.0     1
2        sub-003    PD  1021  PD1021   68      F    26   10.0     1
3        sub-004    PD  1031  PD1031   80      M    22   10.0     1
4        sub-005    PD  1041  PD1041   56      M    21   13.0     1


In [3]:
set_file_path = "/kaggle/input/datasets/adithyarajnarayanan/pd-eeg-dataset-2/PD EEG dataset/sub-001/eeg/sub-001_task-Rest_eeg.set"

# Load the raw EEG data using MNE
raw = mne.io.read_raw_eeglab(set_file_path, preload=True)
eeg_signals = raw.get_data()

print("Shape of EEG signals array (C, L):", eeg_signals.shape)

Reading /kaggle/input/datasets/adithyarajnarayanan/pd-eeg-dataset-2/PD EEG dataset/sub-001/eeg/sub-001_task-Rest_eeg.fdt
Reading 0 ... 140829  =      0.000 ...   281.658 secs...
Shape of EEG signals array (C, L): (63, 140830)


/tmp/ipykernel_98/2417030768.py:4: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_file_path, preload=True)


In [4]:
channel_names = raw.ch_names
print(f"Total number of channels: {len(channel_names)}")
print("Channel names:")
print(", ".join([f"{i+1}: {ch}" for i, ch in enumerate(channel_names)]))

Total number of channels: 63
Channel names:
1: Fp1, 2: Fz, 3: F3, 4: F7, 5: FT9, 6: FC5, 7: FC1, 8: C3, 9: T7, 10: TP9, 11: CP5, 12: CP1, 13: P3, 14: P7, 15: O1, 16: Oz, 17: O2, 18: P4, 19: P8, 20: TP10, 21: CP6, 22: CP2, 23: Cz, 24: C4, 25: T8, 26: FT10, 27: FC6, 28: FC2, 29: F4, 30: F8, 31: Fp2, 32: AF7, 33: AF3, 34: AFz, 35: F1, 36: F5, 37: FT7, 38: FC3, 39: C1, 40: C5, 41: TP7, 42: CP3, 43: P1, 44: P5, 45: PO7, 46: PO3, 47: POz, 48: PO4, 49: PO8, 50: P6, 51: P2, 52: CPz, 53: CP4, 54: TP8, 55: C6, 56: C2, 57: FC4, 58: FT8, 59: F6, 60: AF8, 61: AF4, 62: F2, 63: FCz


In [5]:
mne.set_log_level('ERROR')

base_dir = "/kaggle/input/datasets/adithyarajnarayanan/pd-eeg-dataset-2/PD EEG dataset"
common_channels = None

for sub_id in range(1, 150):
    sub_str = f"sub-{sub_id:03d}"
    set_file_path = os.path.join(base_dir, sub_str, "eeg", f"{sub_str}_task-Rest_eeg.set")
    
    if os.path.exists(set_file_path):
        raw = mne.io.read_raw_eeglab(set_file_path, preload=False, verbose=False)
        raw.rename_channels({ch: ch.strip() for ch in raw.ch_names})
        raw.pick("eeg")
        ch_set = set(raw.ch_names)
        
        if common_channels is None:
            common_channels = ch_set
        else:
            common_channels = common_channels.intersection(ch_set)

# Reset MNE log level back to default if desired
mne.set_log_level('INFO')

common_channels_list = sorted(list(common_channels))
print(f"Total common EEG channels across all subjects: {len(common_channels_list)}")
print("Common channels:")
print(", ".join(common_channels_list))

/tmp/ipykernel_98/331854159.py:11: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_file_path, preload=False, verbose=False)
/tmp/ipykernel_98/331854159.py:11: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_file_path, preload=False, verbose=False)
/tmp/ipykernel_98/331854159.py:11: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_file_path, preload=False, verbose=False)
/tmp/ipykernel_98/331854159.py:11: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_file_path, preloa

Total common EEG channels across all subjects: 60
Common channels:
AF3, AF4, AF7, AF8, AFz, C1, C2, C3, C4, C5, C6, CP1, CP2, CP3, CP4, CP5, CP6, CPz, Cz, F1, F2, F3, F4, F5, F6, F7, F8, FC1, FC2, FC3, FC4, FC5, FC6, FCz, FT10, FT7, FT8, Fp1, Fp2, Fz, O1, O2, Oz, P1, P2, P3, P4, P5, P6, P7, P8, PO7, PO8, POz, T7, T8, TP10, TP7, TP8, TP9


/tmp/ipykernel_98/331854159.py:11: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_file_path, preload=False, verbose=False)
/tmp/ipykernel_98/331854159.py:11: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_file_path, preload=False, verbose=False)


In [6]:
import numpy as np
import mne

def load_segment_set(set_file_path, l_freq, h_freq, target_sfreq=256, window_sec=2, overlap_ratio=0.5, peak_to_peak_threshold=0.00028, truncate_ratio=0.5):
    
    # Load recording (using read_raw_eeglab for .set/.fdt files)
    raw = mne.io.read_raw_eeglab(set_file_path, preload=True, verbose=False)

    # Define the precise channel order requested
    target_channels = common_channels_list

    # Reorder and pick the specific channels
    valid_channels = [ch for ch in target_channels if ch in raw.ch_names]
    raw.pick(valid_channels)

    # 2 & 3. Bandpass Filter (0.5 to 45 Hz)
    raw.filter(l_freq=0.5, h_freq=45.0, fir_design='firwin', verbose=False)

    # 4. Notch Filter at 50 Hz to eliminate line noise
    raw.notch_filter(freqs=50.0, fir_design='firwin', verbose=False)

    # 5. Common Average Reference (CAR)
    raw.set_eeg_reference(ref_channels='average', verbose=False)

    # 6. Resample to target frequency
    raw.resample(target_sfreq, verbose=False)

    # Get data matrix
    signals = raw.get_data()
    
    # Truncate the length of the signal based on the truncate_ratio
    if truncate_ratio < 1.0:
        new_length = int(signals.shape[1] * truncate_ratio)
        signals = signals[:, :new_length]

    print("Signal shape (C, L):", signals.shape)
    C, L = signals.shape
    window_samples = int(window_sec * target_sfreq)

    # Calculate stride samples based on the overlap ratio (e.g., 0.5 means 50% overlap)
    stride_samples = int(window_samples * (1 - overlap_ratio))
    if stride_samples < 1:
        stride_samples = 1

    # 7. Generate sequential windows & Apply Artifact Rejection
    window_list = []
    start = 0
    while start + window_samples <= L:
        end = start + window_samples
        window = signals[:, start:end]
        
        # 8. Peak-to-Peak Threshold Artifact Rejection
        peak_to_peak = np.ptp(window, axis=1)
        if np.any(peak_to_peak > peak_to_peak_threshold):
            start += stride_samples
            continue
        
        window_list.append(window)
        start += stride_samples

    # Check if any windows were created
    if len(window_list) == 0:
        return np.empty((0, C, window_samples))

    # Convert to standard array format (N, C, T)
    windows = np.array(window_list)
    if l_freq > 0 and h_freq > 0:
        windows = mne.filter.filter_data(data=windows, sfreq=target_sfreq, l_freq=l_freq, h_freq=h_freq, method='iir', verbose=False)

    return windows

In [7]:
ids = df.iloc[:,0].values
state = df.iloc[:,1].values 

In [8]:
def get_data(l_freq, h_freq, peak_to_peak_threshold=0.00028):
    X_pd = []
    X_hc = []
    
    # Base directory path for the dataset
    base_dir = "/kaggle/input/datasets/adithyarajnarayanan/pd-eeg-dataset-2/PD EEG dataset"
    
    # Loop through subject indices from 1 to 149
    for sub_id in range(1, 150):
        print("patient number is", sub_id)
        sub_str = f"sub-{sub_id:03d}"
        set_file_path = os.path.join(base_dir, sub_str, "eeg", f"{sub_str}_task-Rest_eeg.set")
        
        # Check if file exists before attempting to load
        if not os.path.exists(set_file_path):
            print(f"File not found for subject {sub_id}")
            continue
            
        if sub_id < 101:
            windows = load_segment_set(
                set_file_path=set_file_path, 
                l_freq=l_freq, 
                h_freq=h_freq, 
                target_sfreq=256, 
                window_sec=2, 
                overlap_ratio=0.0,
                peak_to_peak_threshold=peak_to_peak_threshold
            )
            print(windows.shape)
            if windows.shape[0] > 0:
                X_pd.append(windows)
            else:
                print("no enough windows subject number", sub_id)
                
        else:
            windows = load_segment_set(
                set_file_path=set_file_path, 
                l_freq=l_freq, 
                h_freq=h_freq, 
                target_sfreq=256, 
                window_sec=2, 
                overlap_ratio=0,
                peak_to_peak_threshold=peak_to_peak_threshold
            )
            print(windows.shape)
            if windows.shape[0] > 0:
                X_hc.append(windows)
            else:
                print("no enough windows subject number", sub_id)
                
    return X_hc, X_pd

In [9]:
def balance_matrices_subject_wise(X_list_c0, X_list_c1):
    c0_windows_per_sub = [sub.shape[0] for sub in X_list_c0]
    c1_windows_per_sub = [sub.shape[0] for sub in X_list_c1]
    
    total_c0 = sum(c0_windows_per_sub)
    total_c1 = sum(c1_windows_per_sub)
    
    if total_c0 == total_c1:
        return np.concatenate(X_list_c0, axis=0), np.concatenate(X_list_c1, axis=0)

    if total_c1 > total_c0:
        maj_list = X_list_c1
        maj_counts = np.array(c1_windows_per_sub)
        target_total = total_c0
        is_c1_majority = True
    else:
        maj_list = X_list_c0
        maj_counts = np.array(c0_windows_per_sub)
        target_total = total_c1
        is_c1_majority = False

    num_maj_subs = len(maj_list)
    allocations = np.zeros(num_maj_subs, dtype=int)
    remaining_target = target_total
    active_subs = np.ones(num_maj_subs, dtype=bool)

    while remaining_target > 0 and np.any(active_subs):
        num_active = np.sum(active_subs)
        base_share = remaining_target // num_active
        remainder = remaining_target % num_active
        
        if base_share == 0:
            chosen_indices = np.where(active_subs)[0][:remaining_target]
            for idx in chosen_indices:
                allocations[idx] += 1
            break
            
        for i in range(num_maj_subs):
            if active_subs[i]:
                share = base_share + (1 if remainder > 0 else 0)
                remainder -= 1 if remainder > 0 else 0
                
                available = maj_counts[i] - allocations[i]
                take = min(share, available)
                
                allocations[i] += take
                remaining_target -= take
                
                if allocations[i] == maj_counts[i]:
                    active_subs[i] = False

    processed_maj_list = []
    rng = np.random.default_rng(SEED)
    for i, sub_windows in enumerate(maj_list):
        n_needed = allocations[i]
        if n_needed > 0:
            chosen_indices = rng.choice(sub_windows.shape[0], size=n_needed, replace=False)
            processed_maj_list.append(sub_windows[chosen_indices])
            
    X_processed_maj = np.concatenate(processed_maj_list, axis=0)

    if is_c1_majority:
        return np.concatenate(X_list_c0, axis=0), X_processed_maj
    else:
        return X_processed_maj, np.concatenate(X_list_c1, axis=0)

In [10]:
def scale_data(X_list):
    scaled = []
    for sub in X_list:
        flat = sub.reshape(-1, sub.shape[-1])
        mu = np.mean(flat, axis=0)
        std = np.std(flat, axis=0) + 1e-8
        scaled.append((sub - mu) / std)
    return scaled

In [11]:
class ChannelAttention(layers.Layer):
    def __init__(self, channels):
        super(ChannelAttention, self).__init__()
        self.attn = layers.Dense(channels, activation='softmax')

    def call(self, x):
        avg_pool = tf.reduce_mean(x, axis=1)
        weights = self.attn(avg_pool) 
        weights = tf.expand_dims(weights, 1) 
        return x * weights


In [12]:
class MotionCodeExtended(Model):
    def __init__(self, latent_dim=16, conv_filters=64, dense_units=64):
        super(MotionCodeExtended, self).__init__()
        self.encoder = models.Sequential([
            layers.Permute((2, 1)),
            ChannelAttention(60),
            layers.Conv1D(conv_filters, 16, activation='relu', padding='same'),
            layers.BatchNormalization(),
            layers.MaxPooling1D(2),
            layers.Conv1D(conv_filters // 2, 8, activation='relu', padding='same'),
            layers.GlobalAveragePooling1D(),
            layers.Dense(dense_units, activation='relu')
        ])
        self.fc_mu = layers.Dense(latent_dim)
        
        # Learnable prototypes
        self.pd_prototype = tf.Variable(tf.random.normal([1, latent_dim]), trainable=True)
        self.hc_prototype = tf.Variable(tf.random.normal([1, latent_dim]), trainable=True)

    def call(self, inputs):
        x = self.encoder(inputs)
        z = self.fc_mu(x)
        
        z_norm = tf.math.l2_normalize(z, axis=1)
        pd_norm = tf.math.l2_normalize(self.pd_prototype, axis=1)
        hc_norm = tf.math.l2_normalize(self.hc_prototype, axis=1)
        
        sim_pd = tf.reduce_sum(z_norm * pd_norm, axis=1, keepdims=True)
        sim_hc = tf.reduce_sum(z_norm * hc_norm, axis=1, keepdims=True)
        
        combined = tf.concat([sim_hc, sim_pd], axis=1)
        probs = tf.nn.softmax(combined, axis=1)
        return probs[:, 1:2]

In [13]:
def run_subject_level_mc_cv_optimized(X_healthy, X_pd, SEED=42):
    X_healthy = scale_data(X_healthy)
    X_pd = scale_data(X_pd)
    
    n_hc, n_pd = len(X_healthy), len(X_pd)
    outer_kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
    thresholds = range(65, 95, 5)
    
    hc_splits = list(outer_kf.split(np.arange(n_hc)))
    pd_splits = list(outer_kf.split(np.arange(n_pd)))
    
    total_correct = 0
    total_subjects = 0
    fold_summary_records = []
    
    # Define Hyperparameter Lists for Cartesian Product Grid Search
    param_grid = {
        'lr': [1e-4],
        'batch_size': [ 32],
        'conv_filters': [64],
        'dense_units': [64]
    }
    
    # Generate all possible hyperparameter combinations
    keys = param_grid.keys()
    all_combinations = [dict(zip(keys, combo)) for combo in product(*param_grid.values())]
    
    for fold in range(5):
        print(f"\n========================================")
        print(f"========== OUTER FOLD {fold+1} / 5 ==========")
        print(f"========================================")
        
        hc_train_all, hc_test = hc_splits[fold]
        pd_train_all, pd_test = pd_splits[fold]
        
        # --- INNER LOOP: Grid Search Hyperparameter Optimization ---
        best_score = -1.0
        best_params = None
        best_threshold = 75
        
        hc_inner_splits = list(KFold(n_splits=3, shuffle=True, random_state=SEED).split(hc_train_all))
        pd_inner_splits = list(KFold(n_splits=3, shuffle=True, random_state=SEED).split(pd_train_all))
        
        for params in all_combinations:
            inner_fold_accuracies = []
            inner_fold_thresholds = []
            
            for inner_fold in range(3):
                hc_tr_in_idx, hc_val_in_idx = hc_inner_splits[inner_fold]
                pd_tr_in_idx, pd_val_in_idx = pd_inner_splits[inner_fold]
                
                hc_train_sub = [X_healthy[hc_train_all[i]] for i in hc_tr_in_idx]
                pd_train_sub = [X_pd[pd_train_all[i]] for i in pd_tr_in_idx]
                hc_val_sub = [X_healthy[hc_train_all[i]] for i in hc_val_in_idx]
                pd_val_sub = [X_pd[pd_train_all[i]] for i in pd_val_in_idx]
                
                # Balance classes for inner training
                X_tr_hc_bal, X_tr_pd_bal = balance_matrices_subject_wise(hc_train_sub, pd_train_sub)
                X_inner_train = np.concatenate([X_tr_hc_bal, X_tr_pd_bal], axis=0)
                y_inner_train = np.concatenate([np.zeros(len(X_tr_hc_bal)), np.ones(len(X_tr_pd_bal))], axis=0)
                
                inner_model = MotionCodeExtended(
                    latent_dim=16, 
                    conv_filters=params['conv_filters'], 
                    dense_units=params['dense_units']
                )
                inner_model.compile(
                    optimizer=tf.keras.optimizers.Adam(learning_rate=params['lr']), 
                    loss='binary_crossentropy'
                )
                early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
                inner_model.fit(
                    X_inner_train, y_inner_train, 
                    epochs=40, batch_size=params['batch_size'], 
                    verbose=0, validation_split=0.1, callbacks=[early_stop]
                )
                
                # Tune decision threshold on inner validation set
                val_subjects = hc_val_sub + pd_val_sub
                val_labels = [0]*len(hc_val_sub) + [1]*len(pd_val_sub)
                
                best_t_inner, max_inner_acc = 75, -1.0
                for t in thresholds:
                    t_preds = [1 if (np.mean(inner_model.predict(sub, verbose=0).flatten()) * 100) >= t else 0 for sub in val_subjects]
                    acc = accuracy_score(val_labels, t_preds)
                    if acc > max_inner_acc:
                        max_inner_acc = acc
                        best_t_inner = t
                
                inner_fold_accuracies.append(max_inner_acc)
                inner_fold_thresholds.append(best_t_inner)
            
            mean_inner_acc = np.mean(inner_fold_accuracies)
            if mean_inner_acc > best_score:
                best_score = mean_inner_acc
                best_params = params
                best_threshold = int(np.mean(inner_fold_thresholds))
        
        print(f">> Best Grid Parameters Selected: {best_params} | Threshold: {best_threshold}% (Inner Acc: {best_score:.4f})")
        
        # --- OUTER TRAINING & TESTING ---
        hc_train_final = [X_healthy[i] for i in hc_train_all]
        pd_train_final = [X_pd[i] for i in pd_train_all]
        
        X_tr_hc_final, X_tr_pd_final = balance_matrices_subject_wise(hc_train_final, pd_train_final)
        X_train_final = np.concatenate([X_tr_hc_final, X_tr_pd_final], axis=0)
        y_train_final = np.concatenate([np.zeros(len(X_tr_hc_final)), np.ones(len(X_tr_pd_final))], axis=0)
        
        final_model = MotionCodeExtended(
            latent_dim=16, 
            conv_filters=best_params['conv_filters'], 
            dense_units=best_params['dense_units']
        )
        final_model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=best_params['lr']), 
            loss='binary_crossentropy'
        )
        early_stop_final = callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
        final_model.fit(
            X_train_final, y_train_final, 
            epochs=80, batch_size=best_params['batch_size'], 
            verbose=0, validation_split=0.1, callbacks=[early_stop_final]
        )
        
        # Test evaluation on the 20% held-out outer fold subjects using Majority Voting
        test_subjects = [X_healthy[i] for i in hc_test] + [X_pd[i] for i in pd_test]
        test_labels = [0]*len(hc_test) + [1]*len(pd_test)
        n_hc_test = len(hc_test)
        n_pd_test = len(pd_test)
        
        hc_correct_count = 0
        pd_correct_count = 0
        
        for sub, true_label in zip(test_subjects, test_labels):
            pct_pd = np.mean(final_model.predict(sub, verbose=0).flatten()) * 100
            vote_thresholds = [best_threshold - 5, best_threshold, best_threshold + 5]
            votes = [1 if pct_pd >= t else 0 for t in vote_thresholds]
            pred = 1 if sum(votes) >= 2 else 0
            
            if pred == true_label:
                if true_label == 0:
                    hc_correct_count += 1
                else:
                    pd_correct_count += 1
                    
        fold_total_correct = hc_correct_count + pd_correct_count
        fold_total_subjects = len(test_subjects)
        
        total_correct += fold_total_correct
        total_subjects += fold_total_subjects
        
        fold_summary_records.append({
            'Fold Number': fold + 1,
            'Optimal Hyperparams': str(best_params),
            'Healthy Correct': f"{hc_correct_count}/{n_hc_test}",
            'PD Correct': f"{pd_correct_count}/{n_pd_test}",
            'Total Correct': f"{fold_total_correct}/{fold_total_subjects}"
        })
        
        print(f"Outer Fold {fold+1} Stats -> Healthy: {hc_correct_count}/{n_hc_test} | PD: {pd_correct_count}/{n_pd_test} | Total: {fold_total_correct}/{fold_total_subjects}")

    # Summary Generation
    summary_df = pd.DataFrame(fold_summary_records)
    total_hc_correct = sum(int(x.split('/')[0]) for x in summary_df['Healthy Correct'])
    total_hc_subjects = sum(int(x.split('/')[1]) for x in summary_df['Healthy Correct'])
    
    total_pd_correct = sum(int(x.split('/')[0]) for x in summary_df['PD Correct'])
    total_pd_subjects = sum(int(x.split('/')[1]) for x in summary_df['PD Correct'])
    
    print(f"\n========================================")
    print(f"Healthy Controls Correct: {total_hc_correct}/{total_hc_subjects}")
    print(f"Parkinson's Disease (PD) Correct: {total_pd_correct}/{total_pd_subjects}")
    print(f"Total Combined Correct: {total_correct}/{total_subjects}")
    print("\n--- Nested Cross-Validation Summary ---")
    print(summary_df.to_string(index=False))
    
    return summary_df

In [14]:
print("full signal")
X_hc,X_pd = get_data(-1,-1)
print("healthy size is",len(X_hc))
print("PD size is",len(X_pd))

df_output = run_subject_level_mc_cv_optimized(X_hc, X_pd, SEED=42)

full signal
patient number is 1


/tmp/ipykernel_98/3202470300.py:7: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_file_path, preload=True, verbose=False)


Signal shape (C, L): (60, 36052)
(24, 60, 512)
patient number is 2


/tmp/ipykernel_98/3202470300.py:7: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_file_path, preload=True, verbose=False)


Signal shape (C, L): (60, 41733)
(81, 60, 512)
patient number is 3


...

Outer Fold 4 Stats -> Healthy: 9/10 | PD: 8/20 | Total: 17/30

========== OUTER FOLD 5 / 5 ==========


2026-08-18 06:01:42.540440: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-08-18 06:01:47.625523: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

>> Best Grid Parameters Selected: {'lr': 0.0001, 'batch_size': 32, 'conv_filters': 64, 'dense_units': 64} | Threshold: 65% (Inner Acc: 0.5498)


2026-08-18 06:03:17.752546: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-08-18 06:03:33.919721: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

Outer Fold 5 Stats -> Healthy: 4/9 | PD: 18/19 | Total: 22/28

Healthy Controls Correct: 30/49
Parkinson's Disease (PD) Correct: 72/99
Total Combined Correct: 102/148

--- Nested Cross-Validation Summary ---
 Fold Number                                                     Optimal Hyperparams Healthy Correct PD Correct Total Correct
           1 {'lr': 0.0001, 'batch_size': 32, 'conv_filters': 64, 'dense_units': 64}            6/10      13/20         19/30
           2 {'lr': 0.0001, 'batch_size': 32, 'conv_filters': 64, 'dense_units': 64}            5/10      18/20         23/30
           3 {'lr': 0.0001, 'batch_size': 32, 'conv_filters': 64, 'dense_units': 64}            6/10      15/20         21/30
           4 {'lr': 0.0001, 'batch_size': 32, 'conv_filters': 64, 'dense_units': 64}            9/10       8/20         17/30
           5 {'lr': 0.0001, 'batch_size': 32, 'conv_filters': 64, 'dense_units': 64}             4/9      18/19         22/28


In [15]:
print("alpha signal")
X_hc,X_pd = get_data(8,12)
print("healthy size is",len(X_hc))
print("PD size is",len(X_pd))

df_output = run_subject_level_mc_cv_optimized(X_hc, X_pd, SEED=42)

alpha signal
patient number is 1


/tmp/ipykernel_98/3202470300.py:7: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_file_path, preload=True, verbose=False)


Signal shape (C, L): (60, 36052)
(24, 60, 512)
patient number is 2


/tmp/ipykernel_98/3202470300.py:7: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_file_path, preload=True, verbose=False)


Signal shape (C, L): (60, 41733)
(81, 60, 512)
patient number is 3


...

Outer Fold 4 Stats -> Healthy: 7/10 | PD: 10/20 | Total: 17/30

========== OUTER FOLD 5 / 5 ==========


2026-08-18 06:17:02.167159: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-08-18 06:17:07.257426: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

>> Best Grid Parameters Selected: {'lr': 0.0001, 'batch_size': 32, 'conv_filters': 64, 'dense_units': 64} | Threshold: 65% (Inner Acc: 0.5142)


2026-08-18 06:18:36.980588: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-08-18 06:19:07.433988: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

Outer Fold 5 Stats -> Healthy: 3/9 | PD: 17/19 | Total: 20/28

Healthy Controls Correct: 30/49
Parkinson's Disease (PD) Correct: 66/99
Total Combined Correct: 96/148

--- Nested Cross-Validation Summary ---
 Fold Number                                                     Optimal Hyperparams Healthy Correct PD Correct Total Correct
           1 {'lr': 0.0001, 'batch_size': 32, 'conv_filters': 64, 'dense_units': 64}            5/10      16/20         21/30
           2 {'lr': 0.0001, 'batch_size': 32, 'conv_filters': 64, 'dense_units': 64}            8/10       9/20         17/30
           3 {'lr': 0.0001, 'batch_size': 32, 'conv_filters': 64, 'dense_units': 64}            7/10      14/20         21/30
           4 {'lr': 0.0001, 'batch_size': 32, 'conv_filters': 64, 'dense_units': 64}            7/10      10/20         17/30
           5 {'lr': 0.0001, 'batch_size': 32, 'conv_filters': 64, 'dense_units': 64}             3/9      17/19         20/28


In [16]:
print("beta signal")
X_hc,X_pd = get_data(13,30)
print("healthy size is",len(X_hc))
print("PD size is",len(X_pd))

df_output = run_subject_level_mc_cv_optimized(X_hc, X_pd, SEED=42)

beta signal
patient number is 1


/tmp/ipykernel_98/3202470300.py:7: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_file_path, preload=True, verbose=False)


Signal shape (C, L): (60, 36052)
(24, 60, 512)
patient number is 2


/tmp/ipykernel_98/3202470300.py:7: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_file_path, preload=True, verbose=False)


Signal shape (C, L): (60, 41733)
(81, 60, 512)
patient number is 3


...

Outer Fold 4 Stats -> Healthy: 3/10 | PD: 14/20 | Total: 17/30

========== OUTER FOLD 5 / 5 ==========


2026-08-18 06:32:41.408761: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-08-18 06:32:46.479402: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

>> Best Grid Parameters Selected: {'lr': 0.0001, 'batch_size': 32, 'conv_filters': 64, 'dense_units': 64} | Threshold: 65% (Inner Acc: 0.5023)


2026-08-18 06:34:16.678549: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-08-18 06:34:32.053603: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

Outer Fold 5 Stats -> Healthy: 7/9 | PD: 13/19 | Total: 20/28

Healthy Controls Correct: 21/49
Parkinson's Disease (PD) Correct: 72/99
Total Combined Correct: 93/148

--- Nested Cross-Validation Summary ---
 Fold Number                                                     Optimal Hyperparams Healthy Correct PD Correct Total Correct
           1 {'lr': 0.0001, 'batch_size': 32, 'conv_filters': 64, 'dense_units': 64}            5/10      14/20         19/30
           2 {'lr': 0.0001, 'batch_size': 32, 'conv_filters': 64, 'dense_units': 64}            4/10      19/20         23/30
           3 {'lr': 0.0001, 'batch_size': 32, 'conv_filters': 64, 'dense_units': 64}            2/10      12/20         14/30
           4 {'lr': 0.0001, 'batch_size': 32, 'conv_filters': 64, 'dense_units': 64}            3/10      14/20         17/30
           5 {'lr': 0.0001, 'batch_size': 32, 'conv_filters': 64, 'dense_units': 64}             7/9      13/19         20/28


In [17]:
print("gamma signal")
X_hc,X_pd = get_data(30,100)
print("healthy size is",len(X_hc))
print("PD size is",len(X_pd))

df_output = run_subject_level_mc_cv_optimized(X_hc, X_pd, SEED=42)

gamma signal
patient number is 1


/tmp/ipykernel_98/3202470300.py:7: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_file_path, preload=True, verbose=False)


Signal shape (C, L): (60, 36052)
(24, 60, 512)
patient number is 2


/tmp/ipykernel_98/3202470300.py:7: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_file_path, preload=True, verbose=False)


Signal shape (C, L): (60, 41733)
(81, 60, 512)
patient number is 3


...

Outer Fold 4 Stats -> Healthy: 3/10 | PD: 15/20 | Total: 18/30

========== OUTER FOLD 5 / 5 ==========


2026-08-18 06:47:55.553723: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-08-18 06:48:00.609183: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

>> Best Grid Parameters Selected: {'lr': 0.0001, 'batch_size': 32, 'conv_filters': 64, 'dense_units': 64} | Threshold: 65% (Inner Acc: 0.4760)


2026-08-18 06:49:33.681134: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-08-18 06:49:43.739727: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

Outer Fold 5 Stats -> Healthy: 6/9 | PD: 11/19 | Total: 17/28

Healthy Controls Correct: 26/49
Parkinson's Disease (PD) Correct: 56/99
Total Combined Correct: 82/148

--- Nested Cross-Validation Summary ---
 Fold Number                                                     Optimal Hyperparams Healthy Correct PD Correct Total Correct
           1 {'lr': 0.0001, 'batch_size': 32, 'conv_filters': 64, 'dense_units': 64}            5/10       8/20         13/30
           2 {'lr': 0.0001, 'batch_size': 32, 'conv_filters': 64, 'dense_units': 64}            8/10      12/20         20/30
           3 {'lr': 0.0001, 'batch_size': 32, 'conv_filters': 64, 'dense_units': 64}            4/10      10/20         14/30
           4 {'lr': 0.0001, 'batch_size': 32, 'conv_filters': 64, 'dense_units': 64}            3/10      15/20         18/30
           5 {'lr': 0.0001, 'batch_size': 32, 'conv_filters': 64, 'dense_units': 64}             6/9      11/19         17/28


In [18]:
print("theta signal")
X_hc,X_pd = get_data(4,8)
print("healthy size is",len(X_hc))
print("PD size is",len(X_pd))

df_output = run_subject_level_mc_cv_optimized(X_hc, X_pd, SEED=42)

theta signal
patient number is 1


/tmp/ipykernel_98/3202470300.py:7: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_file_path, preload=True, verbose=False)


Signal shape (C, L): (60, 36052)
(24, 60, 512)
patient number is 2


/tmp/ipykernel_98/3202470300.py:7: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_file_path, preload=True, verbose=False)


Signal shape (C, L): (60, 41733)
(81, 60, 512)
patient number is 3


...

Outer Fold 4 Stats -> Healthy: 9/10 | PD: 12/20 | Total: 21/30

========== OUTER FOLD 5 / 5 ==========


2026-08-18 07:03:03.447420: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-08-18 07:03:11.061948: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

>> Best Grid Parameters Selected: {'lr': 0.0001, 'batch_size': 32, 'conv_filters': 64, 'dense_units': 64} | Threshold: 68% (Inner Acc: 0.5800)


2026-08-18 07:04:51.257986: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


Outer Fold 5 Stats -> Healthy: 7/9 | PD: 10/19 | Total: 17/28

Healthy Controls Correct: 38/49
Parkinson's Disease (PD) Correct: 60/99
Total Combined Correct: 98/148

--- Nested Cross-Validation Summary ---
 Fold Number                                                     Optimal Hyperparams Healthy Correct PD Correct Total Correct
           1 {'lr': 0.0001, 'batch_size': 32, 'conv_filters': 64, 'dense_units': 64}            5/10      15/20         20/30
           2 {'lr': 0.0001, 'batch_size': 32, 'conv_filters': 64, 'dense_units': 64}            7/10      15/20         22/30
           3 {'lr': 0.0001, 'batch_size': 32, 'conv_filters': 64, 'dense_units': 64}           10/10       8/20         18/30
           4 {'lr': 0.0001, 'batch_size': 32, 'conv_filters': 64, 'dense_units': 64}            9/10      12/20         21/30
           5 {'lr': 0.0001, 'batch_size': 32, 'conv_filters': 64, 'dense_units': 64}             7/9      10/19         17/28


In [19]:
print("delta signal")
X_hc,X_pd = get_data(0.5,4)
print("healthy size is",len(X_hc))
print("PD size is",len(X_pd))

df_output = run_subject_level_mc_cv_optimized(X_hc, X_pd, SEED=42)

delta signal
patient number is 1


/tmp/ipykernel_98/3202470300.py:7: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_file_path, preload=True, verbose=False)


Signal shape (C, L): (60, 36052)
(24, 60, 512)
patient number is 2


/tmp/ipykernel_98/3202470300.py:7: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_file_path, preload=True, verbose=False)


Signal shape (C, L): (60, 41733)
(81, 60, 512)
patient number is 3


...

Outer Fold 4 Stats -> Healthy: 3/10 | PD: 12/20 | Total: 15/30

========== OUTER FOLD 5 / 5 ==========


2026-08-18 07:18:20.114021: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-08-18 07:18:25.147417: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

>> Best Grid Parameters Selected: {'lr': 0.0001, 'batch_size': 32, 'conv_filters': 64, 'dense_units': 64} | Threshold: 65% (Inner Acc: 0.4822)


2026-08-18 07:19:57.271742: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-08-18 07:20:08.063427: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

Outer Fold 5 Stats -> Healthy: 7/9 | PD: 6/19 | Total: 13/28

Healthy Controls Correct: 34/49
Parkinson's Disease (PD) Correct: 38/99
Total Combined Correct: 72/148

--- Nested Cross-Validation Summary ---
 Fold Number                                                     Optimal Hyperparams Healthy Correct PD Correct Total Correct
           1 {'lr': 0.0001, 'batch_size': 32, 'conv_filters': 64, 'dense_units': 64}            7/10       9/20         16/30
           2 {'lr': 0.0001, 'batch_size': 32, 'conv_filters': 64, 'dense_units': 64}            9/10       2/20         11/30
           3 {'lr': 0.0001, 'batch_size': 32, 'conv_filters': 64, 'dense_units': 64}            8/10       9/20         17/30
           4 {'lr': 0.0001, 'batch_size': 32, 'conv_filters': 64, 'dense_units': 64}            3/10      12/20         15/30
           5 {'lr': 0.0001, 'batch_size': 32, 'conv_filters': 64, 'dense_units': 64}             7/9       6/19         13/28
